### Importing libraries:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import logging
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import logging
import os
import shutil
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import optuna
from tqdm.auto import tqdm
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
file_path=r"datasets\cleaned_dataset_v4.csv"

df=pd.read_csv(file_path)

c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dropping some initial columns:

I keep only the houses and apartments. I also remove some columns with extremely high percentage of missing rows. For more information, refer to the previous part of the project (EDA)

In [ ]:
# =========================================================
# 1. START FROM df AND KEEP ONLY HOUSE + APARTMENT
# =========================================================
df = df[df["property_type"].isin(["House", "Apartment"])].copy()


cols_to_drop = [
    "parking_places_outdoor", "wash_room", "front_facade_orientation", "diningrooms",
    "parking_places_indoor", "certification_gasoil_tank", "opportunity_for_professional",
    "water_softener", "garden_orientation", "low_energy", "maintenance_cost",
    "terrace_orientation", "security_door", "rain_water_tank", "p_score", "g_score",
    "air_conditioning", "surroundings_protected", "heat_pump", "alarm",
    "terrain_width_roadside", "planning_permission_granted", "frontage_width",
    "solar_panels", "kitchen_type", "garden_surface", "vat", "demarcated_flooding_area",
    "availability", "url", "property_id"
]

df.drop(columns=cols_to_drop, inplace=True, errors="ignore")


### Train and test split:

In [ ]:
from sklearn.model_selection import train_test_split
TEST_SIZE=0.2
RANDOM_STATE=42
train_global, test_global, = train_test_split(
            df,test_size=TEST_SIZE, random_state=RANDOM_STATE
        )
print(f"   ✓ Train: {len(train_global):,} | Test: {len(test_global):,}")

### Splitting by property type (Apartment vs House):


In [ ]:
def split_by_type(train_df, test_df):
    train_apartment = train_df[train_df["property_type"] == "Apartment"]
    test_apartment  = test_df[test_df["property_type"] == "Apartment"]

    train_house = train_df[train_df["property_type"] == "House"]
    test_house  = test_df[test_df["property_type"] == "House"]

    return {
        "train_apartment": train_apartment,
        "test_apartment":  test_apartment,
        "train_house":     train_house,
        "test_house":      test_house
    }


### Splitting by clusters (Luxury vs Residential):

In [1]:

def apply_clustering(
    train_subset,
    test_subset,
    cols_for_clustering=["price", "area", "rooms"], # price is only a feature for clustering- clusters themselves will be dropped later to avoid data leakage
    n_clusters=2,
    random_state=42
):

    # 1. Extract data
    train_cluster_data = train_subset[cols_for_clustering]
    test_cluster_data  = test_subset[cols_for_clustering]

    # 2. Impute only on TRAIN
    imputer = SimpleImputer(strategy="median")
    X_train_imputed = imputer.fit_transform(train_cluster_data)
    X_test_imputed  = imputer.transform(test_cluster_data)

    # 3. Scale only on TRAIN
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)
    X_test_scaled  = scaler.transform(X_test_imputed)

    # 4. KMeans only on TRAIN
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    train_clusters = kmeans.fit_predict(X_train_scaled)
    test_clusters  = kmeans.predict(X_test_scaled)

    # 5. Save clusters back
    train_subset = train_subset.copy()
    test_subset  = test_subset.copy()

    train_subset["cluster"] = train_clusters
    test_subset["cluster"]  = test_clusters

    logger.info(f"[CLUSTER] Train cluster counts: {train_subset['cluster'].value_counts().to_dict()}")
    logger.info(f"[CLUSTER] Test cluster counts:  {test_subset['cluster'].value_counts().to_dict()}")

    return train_subset, test_subset, kmeans, scaler, imputer


### Splitting by both type and cluster:

In [ ]:

def complete_split(
    data,
    do_split_cluster=False,
    do_split_type=False,
    test_size=0.2,
    cols_for_clustering=["price", "area", "rooms"]
):

    # A. Global split
    train_global, test_global = train_test_split(
        data, 
        test_size=test_size, 
        random_state=42
    )

    print(f"[INFO] train_global: {len(train_global)}")
    print(f"[INFO] test_global:  {len(test_global)}")

    # ---------------------------------------------------------------
    # B. OPTIONAL CLUSTERING
    # ---------------------------------------------------------------
    if do_split_cluster:

        train_global, test_global, kmeans, scaler,imputer = apply_clustering(
            train_global,
            test_global,
            cols_for_clustering=cols_for_clustering
        )

        train_luxury= train_global[train_global["cluster"] == 0]
        train_residential= train_global[train_global["cluster"] == 1]

        test_luxury= test_global[test_global["cluster"] == 0]
        test_residential= test_global[test_global["cluster"] == 1]

        print(f"[INFO] train_luxury:      {len(train_luxury)}")
        print(f"[INFO] train_residential: {len(train_residential)}")
        print(f"[INFO] test_luxury:       {len(test_luxury)}")
        print(f"[INFO] test_residential:  {len(test_residential)}")

        # -----------------------------------------------------------
        # C. OPTIONAL TYPE SPLIT inside clusters
        # -----------------------------------------------------------
        if do_split_type:

            train_luxury_apartment  = train_luxury[train_luxury["property_type"] == "Apartment"]
            test_luxury_apartment   = test_luxury[test_luxury["property_type"] == "Apartment"]

            train_residential_apartment  = train_residential[train_residential["property_type"] == "Apartment"]
            test_residential_apartment   = test_residential[test_residential["property_type"] == "Apartment"]

            train_luxury_house = train_luxury[train_luxury["property_type"] == "House"]
            test_luxury_house  = test_luxury[test_luxury["property_type"] == "House"]

            train_residential_house = train_residential[train_residential["property_type"] == "House"]
            test_residential_house  = test_residential[test_residential["property_type"] == "House"]

            print(f"[INFO] train_luxury_apartment:     {len(train_luxury_apartment)}")
            print(f"[INFO] test_luxury_apartment:      {len(test_luxury_apartment)}")
            print(f"[INFO] train_residential_apartment:{len(train_residential_apartment)}")
            print(f"[INFO] test_residential_apartment: {len(test_residential_apartment)}")
            print(f"[INFO] train_luxury_house:         {len(train_luxury_house)}")
            print(f"[INFO] test_luxury_house:          {len(test_luxury_house)}")
            print(f"[INFO] train_residential_house:    {len(train_residential_house)}")
            print(f"[INFO] test_residential_house:     {len(test_residential_house)}")

            return {
                "train_luxury_apartment": train_luxury_apartment,
                "test_luxury_apartment":  test_luxury_apartment,
                "train_residential_apartment": train_residential_apartment,
                "test_residential_apartment":  test_residential_apartment,
                "train_luxury_house":     train_luxury_house,
                "test_luxury_house":      test_luxury_house,
                "train_residential_house":train_residential_house,
                "test_residential_house": test_residential_house,
                "kmeans": kmeans,
                "scaler": scaler
            }

        return {
            "train_luxury":      train_luxury,
            "test_luxury":       test_luxury,
            "train_residential": train_residential,
            "test_residential":  test_residential,
            "kmeans": kmeans,
            "scaler": scaler
        }

    # ---------------------------------------------------------------
    # D. ONLY TYPE SPLIT
    # ---------------------------------------------------------------
    if do_split_type:
        return split_by_type(train_global, test_global)

    # ---------------------------------------------------------------
    # E. GLOBAL RESULT
    # ---------------------------------------------------------------
    return {
        "train_global": train_global,
        "test_global":  test_global
    }



In [ ]:

result_only_type = complete_split(df,do_split_cluster=False,do_split_type=True)

train_apartment = result_only_type["train_apartment"]
test_apartment  = result_only_type["test_apartment"]

train_house = result_only_type["train_house"]
test_house  = result_only_type["test_house"]



result_complete_split = complete_split(df,do_split_cluster=True,do_split_type=False)

train_luxury= result_complete_split["train_luxury"]
test_luxury= result_complete_split["test_luxury"]
train_residential= result_complete_split["train_residential"]
test_residential= result_complete_split["test_residential"]


result_complete_split = complete_split(df,do_split_cluster=True,do_split_type=True)

train_luxury_apartment=result_complete_split["train_luxury_apartment"]
test_luxury_apartment=result_complete_split["test_luxury_apartment"]
train_residential_apartment=result_complete_split["train_residential_apartment"]
test_residential_apartment=result_complete_split["test_residential_apartment"]
train_luxury_house=result_complete_split["train_luxury_house"]
test_luxury_house=result_complete_split["test_luxury_house"]
train_residential_house=result_complete_split["train_residential_house"]
test_residential_house=result_complete_split["test_residential_house"]

split_by_type(train_global,test_global)


## Cleaning datasets:

### Global dataset:

In [ ]:
import pandas as pd

subsets = {
    "train_global":  train_global,
    "test_global":   test_global,

}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df


In [ ]:
# Cleaning global datasets
datasets = [
    (train_global, test_global)     
]




cols_to_drop = [
    "parking_places_outdoor", "wash_room", "front_facade_orientation", "diningrooms",
    "parking_places_indoor", "certification_gasoil_tank", "opportunity_for_professional",
    "water_softener", "garden_orientation", "low_energy", "maintenance_cost",
    "terrace_orientation", "security_door", "rain_water_tank", "p_score", "g_score",
    "air_conditioning", "surroundings_protected", "heat_pump", "alarm",
    "terrain_width_roadside", "planning_permission_granted", "frontage_width",
    "solar_panels", "kitchen_type", "garden_surface", "vat", "demarcated_flooding_area",
    "availability", "url", "property_id", "postal_code"
]

binary_cols = [
    "cellar", "sewer_connection", "has_swimming_pool",
    "preemption_right", "access_disabled", "running_water",
    "is_furnished", "garage", "leased", "has_garden",
    "has_terrace"
]

numeric_keep_missing = [
    "kitchen_surface", "living_room_surface",
    "land_surface", "apartement_floor",
    "number_floors"
]

categorical_keep_missing = [
    "has_equipped_kitchen",
    "glazing_type",
    "heating_type",
    "state",
]

state_grouped_mapping = {
    "To renovate": 0, "To be renovated": 0, "To restore": 0, "To demolish": 0,
    "Under construction": 1,
    "Normal": 2,
    "Fully renovated": 3, "Excellent": 3,
    "New": 4,
    "Missing": -1
}

kitchen_equipped_mapping = {
    "Missing": -1,
    "Not equipped": 0,
    "Partially equipped": 1,
    "Fully equipped": 2,
    "Super equipped": 3
}

glazing_mapping = {
    "Missing": -1,
    "Simple glass": 0,
    "Double glass": 1,
    "Triple glass": 2
}

heating_mapping = {
    "Missing": -1,
    "Not specified": -1,
    "Coal": 0, "Wood": 1, "Fuel oil": 2,
    "Gas": 3, "Hot air": 4,
    "Electricity": 5,
    "Solar energy": 6
}

numeric_key_fields = [
    "primary_energy_consumption", "build_year", "bathrooms",
    "elevator", "facades_number", "area", "rooms", "terrace_surface",
    "co2", "attic", "entry_phone", "cadastral_income",
    "toilets"
]

flooding_mapping = {
    "no flooding area": 1,
    "Low risk": 1,
    "(information not available)": 0
}



for train_df, test_df in datasets:

    # 1. DROP
    train_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

    # 2. BINARY ENCODING
    for col in binary_cols:
        train_df[col] = train_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)
        test_df[col] = test_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)

    # 3. NUMERIC KEEP MISSING = -1
    for col in numeric_keep_missing:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 4. CATEGORICAL – KEEP MISSING AS CATEGORY
    for col in categorical_keep_missing:
        for df in (train_df, test_df):
            df[col] = df[col].astype("category")
            df[col] = df[col].cat.add_categories("Missing")
            df[col] = df[col].fillna("Missing")

    # 5. STATE
    train_df["state"] = train_df["state"].fillna("Missing").map(state_grouped_mapping)
    test_df["state"] = test_df["state"].fillna("Missing").map(state_grouped_mapping)

    # 6. KITCHEN TYPE
    train_df["has_equipped_kitchen"] = (
        train_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )
    test_df["has_equipped_kitchen"] = (
        test_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )

    # 7. GLAZING TYPE
    for df in (train_df, test_df):
        df["glazing_type"] = df["glazing_type"].astype("string").fillna("Missing").map(glazing_mapping)

    # 8. HEATING
    train_df["heating_type"] = train_df["heating_type"].map(heating_mapping)
    test_df["heating_type"] = test_df["heating_type"].map(heating_mapping)

    # 9. certification_electrical_installation
    for df in (train_df, test_df):
        df["certification_electrical_installation"] = (
            df["certification_electrical_installation"]
            .map({
                "yes, certificate in accordance": 1,
                "No, certificate does not comply": 0
            })
            .fillna(-1)
            .astype(int)
        )

    # 10. NUMERIC KEY FIELDS FILLNA = -1
    for col in numeric_key_fields:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 11. flooding_area_type
    for df in (train_df, test_df):
        df["flooding_area_type"] = (
            df["flooding_area_type"]
            .map(flooding_mapping)
            .fillna(-1)
            .astype(int)
        )


In [ ]:
import pandas as pd

subsets = {
    "train_global":  train_global,
    "test_global":   test_global,

}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df


### Luxury cluster:

In [ ]:
import pandas as pd

subsets = {
    "train_luxury":  train_luxury,
    "test_luxury":   test_luxury,

}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df


In [ ]:
# Cleaning datasets
datasets = [
    (train_luxury, test_luxury)     
]




cols_to_drop = [
    "parking_places_outdoor", "wash_room", "front_facade_orientation", "diningrooms",
    "parking_places_indoor", "certification_gasoil_tank", "opportunity_for_professional",
    "water_softener", "garden_orientation", "low_energy", "maintenance_cost",
    "terrace_orientation", "security_door", "rain_water_tank", "p_score", "g_score",
    "air_conditioning", "surroundings_protected", "heat_pump", "alarm",
    "terrain_width_roadside", "planning_permission_granted", "frontage_width",
    "solar_panels", "kitchen_type", "garden_surface", "vat", "demarcated_flooding_area",
    "availability", "url", "property_id", "postal_code", "cluster"
]

binary_cols = [
    "cellar", "sewer_connection", "has_swimming_pool",
    "preemption_right", "access_disabled", "running_water",
    "is_furnished", "garage", "leased", "has_garden",
    "has_terrace"
]

numeric_keep_missing = [
    "kitchen_surface", "living_room_surface",
    "land_surface", "apartement_floor",
    "number_floors"
]

categorical_keep_missing = [
    "has_equipped_kitchen",
    "glazing_type",
    "heating_type",
    "state",
]

state_grouped_mapping = {
    "To renovate": 0, "To be renovated": 0, "To restore": 0, "To demolish": 0,
    "Under construction": 1,
    "Normal": 2,
    "Fully renovated": 3, "Excellent": 3,
    "New": 4,
    "Missing": -1
}

kitchen_equipped_mapping = {
    "Missing": -1,
    "Not equipped": 0,
    "Partially equipped": 1,
    "Fully equipped": 2,
    "Super equipped": 3
}

glazing_mapping = {
    "Missing": -1,
    "Simple glass": 0,
    "Double glass": 1,
    "Triple glass": 2
}

heating_mapping = {
    "Missing": -1,
    "Not specified": -1,
    "Coal": 0, "Wood": 1, "Fuel oil": 2,
    "Gas": 3, "Hot air": 4,
    "Electricity": 5,
    "Solar energy": 6
}

numeric_key_fields = [
    "primary_energy_consumption", "build_year", "bathrooms",
    "elevator", "facades_number", "area", "rooms", "terrace_surface",
    "co2", "attic", "entry_phone", "cadastral_income",
    "toilets"
]

flooding_mapping = {
    "no flooding area": 1,
    "Low risk": 1,
    "(information not available)": 0
}



for train_df, test_df in datasets:

    # 1. DROP
    train_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

    # 2. BINARY ENCODING
    for col in binary_cols:
        train_df[col] = train_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)
        test_df[col] = test_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)

    # 3. NUMERIC KEEP MISSING = -1
    for col in numeric_keep_missing:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 4. CATEGORICAL – KEEP MISSING AS CATEGORY
    for col in categorical_keep_missing:
        for df in (train_df, test_df):
            df[col] = df[col].astype("category")
            df[col] = df[col].cat.add_categories("Missing")
            df[col] = df[col].fillna("Missing")

    # 5. STATE
    train_df["state"] = train_df["state"].fillna("Missing").map(state_grouped_mapping)
    test_df["state"] = test_df["state"].fillna("Missing").map(state_grouped_mapping)

    # 6. KITCHEN TYPE
    train_df["has_equipped_kitchen"] = (
        train_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )
    test_df["has_equipped_kitchen"] = (
        test_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )

    # 7. GLAZING TYPE
    for df in (train_df, test_df):
        df["glazing_type"] = df["glazing_type"].astype("string").fillna("Missing").map(glazing_mapping)

    # 8. HEATING
    train_df["heating_type"] = train_df["heating_type"].map(heating_mapping)
    test_df["heating_type"] = test_df["heating_type"].map(heating_mapping)

    # 9. certification_electrical_installation
    for df in (train_df, test_df):
        df["certification_electrical_installation"] = (
            df["certification_electrical_installation"]
            .map({
                "yes, certificate in accordance": 1,
                "No, certificate does not comply": 0
            })
            .fillna(-1)
            .astype(int)
        )

    # 10. NUMERIC KEY FIELDS FILLNA = -1
    for col in numeric_key_fields:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 11. flooding_area_type
    for df in (train_df, test_df):
        df["flooding_area_type"] = (
            df["flooding_area_type"]
            .map(flooding_mapping)
            .fillna(-1)
            .astype(int)
        )


In [ ]:
import pandas as pd

subsets = {
    "train_luxury":  train_luxury,
    "test_luxury":   test_luxury,

}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df


### Residential cluster:


In [ ]:
import pandas as pd

subsets = {
    "train_residential":  train_residential,
    "test_residential":   test_residential,

}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df


In [ ]:
# Cleaning datasets
datasets = [
    (train_residential, test_residential)     
]



cols_to_drop = [
    "parking_places_outdoor", "wash_room", "front_facade_orientation", "diningrooms",
    "parking_places_indoor", "certification_gasoil_tank", "opportunity_for_professional",
    "water_softener", "garden_orientation", "low_energy", "maintenance_cost",
    "terrace_orientation", "security_door", "rain_water_tank", "p_score", "g_score",
    "air_conditioning", "surroundings_protected", "heat_pump", "alarm",
    "terrain_width_roadside", "planning_permission_granted", "frontage_width",
    "solar_panels", "kitchen_type", "garden_surface", "vat", "demarcated_flooding_area",
    "availability", "url", "property_id", "postal_code", "cluster"
]

binary_cols = [
    "cellar", "sewer_connection", "has_swimming_pool",
    "preemption_right", "access_disabled", "running_water",
    "is_furnished", "garage", "leased", "has_garden",
    "has_terrace"
]

numeric_keep_missing = [
    "kitchen_surface", "living_room_surface",
    "land_surface", "apartement_floor",
    "number_floors"
]

categorical_keep_missing = [
    "has_equipped_kitchen",
    "glazing_type",
    "heating_type",
    "state",
]

state_grouped_mapping = {
    "To renovate": 0, "To be renovated": 0, "To restore": 0, "To demolish": 0,
    "Under construction": 1,
    "Normal": 2,
    "Fully renovated": 3, "Excellent": 3,
    "New": 4,
    "Missing": -1
}

kitchen_equipped_mapping = {
    "Missing": -1,
    "Not equipped": 0,
    "Partially equipped": 1,
    "Fully equipped": 2,
    "Super equipped": 3
}

glazing_mapping = {
    "Missing": -1,
    "Simple glass": 0,
    "Double glass": 1,
    "Triple glass": 2
}

heating_mapping = {
    "Missing": -1,
    "Not specified": -1,
    "Coal": 0, "Wood": 1, "Fuel oil": 2,
    "Gas": 3, "Hot air": 4,
    "Electricity": 5,
    "Solar energy": 6
}

numeric_key_fields = [
    "primary_energy_consumption", "build_year", "bathrooms",
    "elevator", "facades_number", "area", "rooms", "terrace_surface",
    "co2", "attic", "entry_phone", "cadastral_income",
    "toilets"
]

flooding_mapping = {
    "no flooding area": 1,
    "Low risk": 1,
    "(information not available)": 0
}


for train_df, test_df in datasets:

    # 1. DROP
    train_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

    # 2. BINARY ENCODING
    for col in binary_cols:
        train_df[col] = train_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)
        test_df[col] = test_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)

    # 3. NUMERIC KEEP MISSING = -1
    for col in numeric_keep_missing:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 4. CATEGORICAL – KEEP MISSING AS CATEGORY
    for col in categorical_keep_missing:
        for df in (train_df, test_df):
            df[col] = df[col].astype("category")
            df[col] = df[col].cat.add_categories("Missing")
            df[col] = df[col].fillna("Missing")

    # 5. STATE
    train_df["state"] = train_df["state"].fillna("Missing").map(state_grouped_mapping)
    test_df["state"] = test_df["state"].fillna("Missing").map(state_grouped_mapping)

    # 6. KITCHEN TYPE
    train_df["has_equipped_kitchen"] = (
        train_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )
    test_df["has_equipped_kitchen"] = (
        test_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )

    # 7. GLAZING TYPE
    for df in (train_df, test_df):
        df["glazing_type"] = df["glazing_type"].astype("string").fillna("Missing").map(glazing_mapping)

    # 8. HEATING
    train_df["heating_type"] = train_df["heating_type"].map(heating_mapping)
    test_df["heating_type"] = test_df["heating_type"].map(heating_mapping)

    # 9. certification_electrical_installation
    for df in (train_df, test_df):
        df["certification_electrical_installation"] = (
            df["certification_electrical_installation"]
            .map({
                "yes, certificate in accordance": 1,
                "No, certificate does not comply": 0
            })
            .fillna(-1)
            .astype(int)
        )

    # 10. NUMERIC KEY FIELDS FILLNA = -1
    for col in numeric_key_fields:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 11. flooding_area_type
    for df in (train_df, test_df):
        df["flooding_area_type"] = (
            df["flooding_area_type"]
            .map(flooding_mapping)
            .fillna(-1)
            .astype(int)
        )


In [ ]:
import pandas as pd

subsets = {
    "train_residential":  train_residential,
    "test_residential":   test_residential,

}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df


### Cleaning houses for residential and luxury clusters:


In [ ]:
import pandas as pd

subsets = {
    "train_residential_house":  train_residential_house,
    "test_residential_house":   test_residential_house,
    "train_luxury_house":  train_luxury_house,
    "test_luxury_house":   test_luxury_house,
}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df

In [ ]:

datasets = [
    (train_luxury_house, test_luxury_house),
    (train_residential_house, test_residential_house),
]





cols_to_drop = [
    "parking_places_outdoor", "wash_room", "front_facade_orientation", "diningrooms",
    "parking_places_indoor", "certification_gasoil_tank", "opportunity_for_professional",
    "water_softener", "garden_orientation", "low_energy", "maintenance_cost",
    "terrace_orientation", "security_door", "rain_water_tank", "p_score", "g_score",
    "air_conditioning", "surroundings_protected", "heat_pump", "alarm",
    "terrain_width_roadside", "planning_permission_granted", "frontage_width",
    "solar_panels", "kitchen_type", "garden_surface", "vat", "demarcated_flooding_area",
    "availability", "url", "property_id","apartement_floor","entry_phone", "postal_code", "cluster"
]

binary_cols = [
    "cellar", "sewer_connection", "has_swimming_pool",
    "preemption_right", "access_disabled", "running_water",
    "is_furnished", "garage", "leased", "has_garden",
    "has_terrace"
]

numeric_keep_missing = [
    "kitchen_surface", "living_room_surface",
    "land_surface",
    "number_floors"
]

categorical_keep_missing = [
    "has_equipped_kitchen",
    "glazing_type",
    "heating_type",
    "state",
]

state_grouped_mapping = {
    "To renovate": 0, "To be renovated": 0, "To restore": 0, "To demolish": 0,
    "Under construction": 1,
    "Normal": 2,
    "Fully renovated": 3, "Excellent": 3,
    "New": 4,
    "Missing": -1
}

kitchen_equipped_mapping = {
    "Missing": -1,
    "Not equipped": 0,
    "Partially equipped": 1,
    "Fully equipped": 2,
    "Super equipped": 3
}

glazing_mapping = {
    "Missing": -1,
    "Simple glass": 0,
    "Double glass": 1,
    "Triple glass": 2
}

heating_mapping = {
    "Missing": -1,
    "Not specified": -1,
    "Coal": 0, "Wood": 1, "Fuel oil": 2,
    "Gas": 3, "Hot air": 4,
    "Electricity": 5,
    "Solar energy": 6
}

numeric_key_fields = [
    "primary_energy_consumption", "build_year", "bathrooms",
    "elevator", "facades_number", "area", "rooms", "terrace_surface",
    "co2", "attic", "cadastral_income",
    "toilets"
]

flooding_mapping = {
    "no flooding area": 1,
    "Low risk": 1,
    "(information not available)": 0
}


for train_df, test_df in datasets:

    # 1. DROP
    train_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

    # 2. BINARY ENCODING
    for col in binary_cols:
        train_df[col] = train_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)
        test_df[col] = test_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)

    # 3. NUMERIC KEEP MISSING = -1
    for col in numeric_keep_missing:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 4. CATEGORICAL – KEEP MISSING AS CATEGORY
    for col in categorical_keep_missing:
        for df in (train_df, test_df):
            df[col] = df[col].astype("category")
            df[col] = df[col].cat.add_categories("Missing")
            df[col] = df[col].fillna("Missing")

    # 5. STATE
    train_df["state"] = train_df["state"].fillna("Missing").map(state_grouped_mapping)
    test_df["state"] = test_df["state"].fillna("Missing").map(state_grouped_mapping)

    # 6. KITCHEN TYPE
    train_df["has_equipped_kitchen"] = (
        train_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )
    test_df["has_equipped_kitchen"] = (
        test_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )

    # 7. GLAZING TYPE
    for df in (train_df, test_df):
        df["glazing_type"] = df["glazing_type"].astype("string").fillna("Missing").map(glazing_mapping)

    # 8. HEATING
    train_df["heating_type"] = train_df["heating_type"].map(heating_mapping)
    test_df["heating_type"] = test_df["heating_type"].map(heating_mapping)

    # 9. certification_electrical_installation
    for df in (train_df, test_df):
        df["certification_electrical_installation"] = (
            df["certification_electrical_installation"]
            .map({
                "yes, certificate in accordance": 1,
                "No, certificate does not comply": 0
            })
            .fillna(-1)
            .astype(int)
        )

    # 10. NUMERIC KEY FIELDS FILLNA = -1
    for col in numeric_key_fields:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 11. flooding_area_type
    for df in (train_df, test_df):
        df["flooding_area_type"] = (
            df["flooding_area_type"]
            .map(flooding_mapping)
            .fillna(-1)
            .astype(int)
        )


In [ ]:
import pandas as pd

subsets = {
    "train_residential_house":  train_residential_house,
    "test_residential_house":   test_residential_house,
    "train_luxury_house":  train_luxury_house,
    "test_luxury_house":   test_luxury_house,
}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df

### Cleaning apartments for residential and luxury clusters:

In [ ]:
import pandas as pd

subsets = {
    "train_residential_apartment":  train_residential_apartment,
    "test_residential_apartment":   test_residential_apartment,
    "train_luxury_apartment":  train_luxury_apartment,
    "test_luxury_apartment":   test_luxury_apartment,
}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df

In [ ]:

datasets = [
    (train_luxury_apartment, test_luxury_apartment),
    (train_residential_apartment, test_residential_apartment),
]





cols_to_drop = [
    "parking_places_outdoor", "wash_room", "front_facade_orientation", "diningrooms",
    "parking_places_indoor", "certification_gasoil_tank", "opportunity_for_professional",
    "water_softener", "garden_orientation", "low_energy", "maintenance_cost",
    "terrace_orientation", "security_door", "rain_water_tank", "p_score", "g_score",
    "air_conditioning", "surroundings_protected", "heat_pump", "alarm",
    "terrain_width_roadside", "planning_permission_granted", "frontage_width",
    "solar_panels", "kitchen_type", "garden_surface", "vat", "demarcated_flooding_area",
    "availability", "url", "property_id","attic", "land_surface", "kitchen_surface",
    "cadastral_income","postal_code", "cluster"
]

binary_cols = [
    "cellar", "sewer_connection", "has_swimming_pool",
    "preemption_right", "access_disabled", "running_water",
    "is_furnished", "garage", "leased", "has_garden",
    "has_terrace"
]

numeric_keep_missing = [ "living_room_surface",
    "number_floors","apartement_floor"
]

categorical_keep_missing = [
    "has_equipped_kitchen",
    "glazing_type",
    "heating_type",
    "state",
]

state_grouped_mapping = {
    "To renovate": 0, "To be renovated": 0, "To restore": 0, "To demolish": 0,
    "Under construction": 1,
    "Normal": 2,
    "Fully renovated": 3, "Excellent": 3,
    "New": 4,
    "Missing": -1
}

kitchen_equipped_mapping = {
    "Missing": -1,
    "Not equipped": 0,
    "Partially equipped": 1,
    "Fully equipped": 2,
    "Super equipped": 3
}

glazing_mapping = {
    "Missing": -1,
    "Simple glass": 0,
    "Double glass": 1,
    "Triple glass": 2
}

heating_mapping = {
    "Missing": -1,
    "Not specified": -1,
    "Coal": 0, "Wood": 1, "Fuel oil": 2,
    "Gas": 3, "Hot air": 4,
    "Electricity": 5,
    "Solar energy": 6
}

numeric_key_fields = [
    "primary_energy_consumption", "build_year", "bathrooms",
    "elevator", "facades_number", "area", "rooms", "terrace_surface",
    "co2",
    "toilets","entry_phone"
]

flooding_mapping = {
    "no flooding area": 1,
    "Low risk": 1,
    "(information not available)": 0
}



for train_df, test_df in datasets:

    # 1. DROP
    train_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    test_df.drop(columns=cols_to_drop, inplace=True, errors="ignore")

    # 2. BINARY ENCODING
    for col in binary_cols:
        train_df[col] = train_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)
        test_df[col] = test_df[col].map({1.0: 1, 0.0: 0}).fillna(-1).astype(int)

    # 3. NUMERIC KEEP MISSING = -1
    for col in numeric_keep_missing:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 4. CATEGORICAL – KEEP MISSING AS CATEGORY
    for col in categorical_keep_missing:
        for df in (train_df, test_df):
            df[col] = df[col].astype("category")
            df[col] = df[col].cat.add_categories("Missing")
            df[col] = df[col].fillna("Missing")

    # 5. STATE
    train_df["state"] = train_df["state"].fillna("Missing").map(state_grouped_mapping)
    test_df["state"] = test_df["state"].fillna("Missing").map(state_grouped_mapping)

    # 6. KITCHEN TYPE
    train_df["has_equipped_kitchen"] = (
        train_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )
    test_df["has_equipped_kitchen"] = (
        test_df["has_equipped_kitchen"].astype("string").fillna("Missing").map(kitchen_equipped_mapping)
    )

    # 7. GLAZING TYPE
    for df in (train_df, test_df):
        df["glazing_type"] = df["glazing_type"].astype("string").fillna("Missing").map(glazing_mapping)

    # 8. HEATING
    train_df["heating_type"] = train_df["heating_type"].map(heating_mapping)
    test_df["heating_type"] = test_df["heating_type"].map(heating_mapping)

    # 9. certification_electrical_installation
    for df in (train_df, test_df):
        df["certification_electrical_installation"] = (
            df["certification_electrical_installation"]
            .map({
                "yes, certificate in accordance": 1,
                "No, certificate does not comply": 0
            })
            .fillna(-1)
            .astype(int)
        )

    # 10. NUMERIC KEY FIELDS FILLNA = -1
    for col in numeric_key_fields:
        train_df[col] = train_df[col].fillna(-1)
        test_df[col] = test_df[col].fillna(-1)

    # 11. flooding_area_type
    for df in (train_df, test_df):
        df["flooding_area_type"] = (
            df["flooding_area_type"]
            .map(flooding_mapping)
            .fillna(-1)
            .astype(int)
        )


In [ ]:
import pandas as pd

subsets = {
    "train_residential_apartment":  train_residential_apartment,
    "test_residential_apartment":   test_residential_apartment,
    "train_luxury_apartment":  train_luxury_apartment,
    "test_luxury_apartment":   test_luxury_apartment,
}

nan_summary = {}

for name, df_subset in subsets.items():
    nan_summary[name] = (df_subset.isnull().sum() * 100 / df_subset.shape[0])

nan_summary_df = pd.DataFrame(nan_summary)
nan_summary_df

### Exporting datasets to csv files:

In [ ]:
datasets = {
    "train_global": train_global,
    "test_global": test_global,
    "train_luxury": train_luxury,
    "test_luxury": test_luxury,
    "train_residential": train_residential,
    "test_residential": test_residential,
    "train_luxury_apartment": train_luxury_apartment,
    "test_luxury_apartment": test_luxury_apartment,
    "train_luxury_house": train_luxury_house,
    "test_luxury_house": test_luxury_house,
    "train_residential_apartment": train_residential_apartment,
    "test_residential_apartment": test_residential_apartment,
    "train_residential_house": train_residential_house,
    "test_residential_house": test_residential_house,
    "train_apartment": train_apartment,
    "test_apartment": test_apartment,
    "train_house": train_house,
    "test_house": test_house
}

for name, df in datasets.items():
    df.to_csv(f"{name}.csv", index=False)
    shutil.move(f"{name}.csv", r"datasets/")


## Training initial model:


### Initial parameters for first assessment

In [ ]:
def choose_params(model_name, n_rows):
  


    if n_rows <= 1500:
        size = "small"
    elif n_rows <= 9000:
        size = "medium"
    else:
        size = "large"

#XGBOOST
    if model_name == "xgb":
        if size == "small":
            return dict(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.08,
                subsample=0.9,
                colsample_bytree=0.9,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=12
            )
        elif size == "medium":
            return dict(
                n_estimators=600,
                max_depth=6,
                learning_rate=0.06,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=12
            )
        else:  # large
            return dict(
                n_estimators=800,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=42,
                n_jobs=12
            )

    # ============================
    # LIGHTGBM
    # ============================
    if model_name == "lgbm":
        if size == "small":
            return dict(
                n_estimators=300,
                learning_rate=0.08,
                max_depth=-1,
                num_leaves=40,
                subsample=0.9,
                colsample_bytree=0.9,
                n_jobs=12
            )
        elif size == "medium":
            return dict(
                n_estimators=600,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=80,
                subsample=0.8,
                colsample_bytree=0.8,
                n_jobs=12
            )
        else:  # large
            return dict(
                n_estimators=900,
                learning_rate=0.04,
                max_depth=-1,
                num_leaves=127,
                subsample=0.8,
                colsample_bytree=0.8,
                n_jobs=12
            )

    # ============================
    # CATBOOST
    # ============================
    if model_name == "cat":
        if size == "small":
            return dict(
                iterations=300,
                depth=6,
                learning_rate=0.08,
                loss_function="RMSE",
                task_type="CPU",
                verbose=False,
                thread_count=12
            )
        elif size == "medium":
            return dict(
                iterations=600,
                depth=8,
                learning_rate=0.06,
                loss_function="RMSE",
                task_type="CPU",
                verbose=False,
                thread_count=12
            )
        else:  # large
            return dict(
                iterations=900,
                depth=10,
                learning_rate=0.04,
                loss_function="RMSE",
                task_type="CPU",
                verbose=False,
                thread_count=12
            )


### Training function:

In [ ]:


def build_preprocessor(X):
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object"]).columns

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    return preprocess


def evaluate_log_model(model, X_test, y_test_real):
    preds_log = model.predict(X_test)
    preds = np.exp(preds_log)

    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_test_real, preds))),
        "MAE": float(mean_absolute_error(y_test_real, preds)),
        "R2":  float(r2_score(y_test_real, preds))
    }


def train_all_models(datasets_dict):
    os.makedirs("models_before_optimisation", exist_ok=True)
    results = []

    for dataset_name, (train_df, test_df) in datasets_dict.items():

        # clean price
        train_df = train_df[train_df["price"] > 0].copy()
        test_df  = test_df[test_df["price"] > 0].copy()

        train_df["log_price"] = np.log(train_df["price"])
        test_df["log_price"]  = np.log(test_df["price"])

        X_train = train_df.drop(columns=["price", "log_price"])
        y_train = train_df["log_price"]

        X_test = test_df.drop(columns=["price", "log_price"])
        y_test = test_df["price"]

        preprocess = build_preprocessor(X_train)
        n_rows = len(train_df)

        # models to run
        model_names = ["xgb", "lgbm", "cat"]

        for model_name in model_names:
            params = choose_params(model_name, n_rows)

            print(f"\n===== Training {model_name.upper()} on {dataset_name} ({n_rows} rows) =====")
            print("Used params:", params)

            if model_name == "xgb":
                model = xgb.XGBRegressor(**params)

            elif model_name == "lgbm":
                model = lgb.LGBMRegressor(**params)

            elif model_name == "cat":
                model = CatBoostRegressor(**params)

            pipeline = Pipeline([
                ("preprocess", preprocess),
                ("model", model)
            ])

            pipeline.fit(X_train, y_train)

            scores = evaluate_log_model(pipeline, X_test, y_test)

            save_path = f"models_before_optimisation/{model_name}_{dataset_name}.pkl"
            with open(save_path, "wb") as f:
                pickle.dump(pipeline, f)



def append_result_row(row: dict, path="results_all_models_before_optimisation.csv"):
    df_row = pd.DataFrame([row])

    if not os.path.exists(path):
        df_row.to_csv(path, index=False)
    else:
        df_row.to_csv(path, mode="a", header=False, index=False)

        row = {
            "dataset": dataset_name,
            "model": model_name,
            "RMSE": scores["RMSE"],
            "MAE": scores["MAE"],
            "R2": scores["R2"],
            "model_path": save_path
        }

        append_result_row(row)

    df_results = pd.DataFrame(results)
    df_results.to_csv("results_all_models_before_optimisation.csv", index=False)
    print("\nSaved results_all_models_before_optimisation.csv")

    return df_results


### Training the models

In [ ]:
datasets = {
    "global": (train_global, test_global),
    "luxury": (train_luxury, test_luxury),
    "residential": (train_residential, test_residential),
    "lux_apt": (train_luxury_apartment, test_luxury_apartment),
    "lux_house": (train_luxury_house, test_luxury_house),
    "res_apt": (train_residential_apartment, test_residential_apartment),
    "res_house": (train_residential_house, test_residential_house)
}

train_all_models(datasets)


The quality of the model trained on the full (global) dataset is too poor to be used, as are the models trained on the luxury clusters. It is natural that investments into highly expensive apartments, as well as , for instance, entire building blocks with 1000s of apartments inside- requires different analytical approach.Given the time constraints, I decided to focus only on the Residential cluster (for both houses and apartments). The deployed model will therefore only be applicable to properties within a certain limit of area and number of rooms. For properties outside of this limit, the monit will inform the user to contact ImmoEliza for a manual quotation. This might get expanded in the future.


The models will be further optimised using optuna to find the best parameters to improve accuracy and reduce overfitting. The best candidate is Catboost, but all three models (XGB,LGBM and Catboost ) will be optimised- as their metrics are quite similar, optimisation might actually swap this trend.

Refer to optimisation.ipynb and optimum_training.ipynb